# Embed Infocom.am Investigation Chunks

Embeds article chunks using **Metric-AI/armenian-text-embeddings-2-large** (1024d) on a free T4 GPU.

**Steps:**
1. Run all cells (repo is cloned automatically)
2. Copy output files from `scraped_data/` to your local machine
3. Run `python load_embeddings.py --strategy paragraph` locally

**Runtime:** ~2 min for 421 paragraph chunks on T4 GPU

In [ ]:
print(509)

In [ ]:
# Install dependencies + clone repo
!pip install -q transformers
!git clone https://github.com/HaykTarkhanyan/rag_workflow.git
%cd rag_workflow

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Load chunks from cloned repo
import json

STRATEGY = "paragraph"  # Change to "sentence" if needed

filename = f"scraped_data/chunks_{STRATEGY}.json"
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]
print(f"Loaded {len(chunks)} chunks ({STRATEGY} strategy)")
print(f"Sample chunk_id: {chunks[0]['chunk_id']}")

In [ ]:
# Load chunks
STRATEGY = "paragraph"  # Change to "sentence" if you uploaded that file

filename = f"chunks_{STRATEGY}.json"
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]
print(f"Loaded {len(chunks)} chunks ({STRATEGY} strategy)")
print(f"Sample chunk_id: {chunks[0]['chunk_id']}")

In [ ]:
# Load model
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import time
import numpy as np

MODEL_NAME = "Metric-AI/armenian-text-embeddings-2-large"
BATCH_SIZE = 32  # GPU can handle larger batches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# Embedding function
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def embed_texts(texts, tokenizer, model, device, batch_size=BATCH_SIZE):
    all_embeddings = []
    t0 = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        batch_dict = tokenizer(
            batch, max_length=512, padding=True, truncation=True, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**batch_dict)

        embeddings = average_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy())

        done = min(i + batch_size, len(texts))
        elapsed = time.time() - t0
        rate = done / elapsed
        eta = (len(texts) - done) / rate if rate > 0 else 0
        print(f"  [{done}/{len(texts)}] {elapsed:.1f}s elapsed, ~{eta:.0f}s remaining")

    return np.vstack(all_embeddings)

print("Embedding function ready")

In [ ]:
# Embed all chunks
texts = [c["text_for_embedding"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

print(f"Embedding {len(texts)} chunks...")
t0 = time.time()
embeddings = embed_texts(texts, tokenizer, model, device)
total = time.time() - t0
print(f"\nDone! {len(texts)} chunks in {total:.1f}s ({total/len(texts)*1000:.0f}ms/chunk)")
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# Quick sanity check: similarity between first 2 chunks of same article
sim = np.dot(embeddings[0], embeddings[1])
print(f"Similarity between chunk 0 and 1 (same article): {sim:.4f}")

# Find most different chunk from chunk 0
sims = embeddings @ embeddings[0]
most_different = np.argmin(sims)
print(f"Most different from chunk 0: chunk {most_different} (sim={sims[most_different]:.4f})")
print(f"  -> {chunks[most_different]['metadata']['title'][:70]}")

In [ ]:
# Download files (if not using VS Code extension to sync)
# Uncomment these lines if you want to download manually:
# from google.colab import files
# files.download(f"{output_prefix}.npy")
# files.download(f"{output_prefix}_meta.json")

# If using VS Code extension, the files are already in scraped_data/
print(f"Files saved in: {output_prefix}.npy and {output_prefix}_meta.json")
print("Copy them to your local scraped_data/ folder, then run:")
print("  python load_embeddings.py --strategy", STRATEGY)

In [ ]:
# Download files to your local machine
files.download(f"{output_prefix}.npy")
files.download(f"{output_prefix}_meta.json")

## Next steps (on local machine)

1. Place downloaded files in `scraped_data/`
2. Run: `python load_embeddings.py --strategy paragraph`
   - This loads the `.npy` + metadata into ChromaDB

Or re-run this notebook with `STRATEGY = "sentence"` for the sentence chunks.